<a href="https://colab.research.google.com/github/Kartik210-5/Small-Language-Model/blob/main/Small_Language_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Importing

In [ ]:
from datasets import load_dataset
import re
import sentencepiece as spm
import os
import torch
import random
import json
import numpy as np
import cupy as cp

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
folder_path = '/content/drive/MyDrive/SLM'

Mounted at /content/drive


#Importing Datasets

In [ ]:
df = load_dataset("roneneldan/TinyStories", split = "train")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
df = df.shuffle(seed=42)
split = df.train_test_split(test_size=0.2)

train = split["train"]
temp = split["test"]

temp = temp.shuffle(seed=42)
splitt = temp.train_test_split(test_size=0.5)

validation = splitt["train"]
test = splitt["test"]

In [ ]:
print("Train: ", train, "\nValidation: ",validation, "\nTest: ", test)

Train:  Dataset({
    features: ['text'],
    num_rows: 1695775
}) 
Validation:  Dataset({
    features: ['text'],
    num_rows: 211972
}) 
Test:  Dataset({
    features: ['text'],
    num_rows: 211972
})


In [ ]:
trainlst = train['text']
validationlst = validation['text']
testlst = test['text']

In [ ]:
dd = load_dataset("DeepPavlov/daily_dialog")

In [ ]:
# Assuming your DatasetDict is stored in a variable named 'dataset'

flattened_dialogues = {}

for split in dd.keys():
    # This loops through every row's dialogue list and flattens them into one single list of strings
    flattened_dialogues[split] = [
        utterance
        for dialogue_list in dd[split]['dialog']
        for utterance in dialogue_list
    ]

# --- Verification ---
for split, dialogues in flattened_dialogues.items():
    print(f"{split.capitalize()} individual utterances: {len(dialogues)}")


Train individual utterances: 474601
Validation individual utterances: 44126
Test individual utterances: 41206


In [ ]:
print("\nFirst 3 training utterances:")
print(flattened_dialogues['validation'][:3])


First 3 training utterances:
['Hello . Capital Hotel . May I help you ? ', ' Yes , unlikely my flight will be 2 hours due to the fog . Would you please keep my reservation ? ', ' Sure . May I have your name please ? ']


#Cleaning Dataset

In [ ]:
def cleans(text):
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s.!?;:\'\"()\-\n]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [ ]:
dtst = folder_path + "/Datasets"

def append_jsonl(the_file, lst):
    i = 0
    with open(dtst + f'/{the_file}.jsonl', "a", encoding="utf-8") as f:
        for article in lst:
            temp = cleans(article)
            i+=1
            if i % 50000 == 0:
              print(f"Processed {i:,} articles")
            record = {"text": temp}
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

# from itertools import islice

# # =========================
# # Regexes (compile once)
# # =========================

# URL_RE = re.compile(r'http\S+|www\S+')
# SPECIAL_RE = re.compile(r'[^a-zA-Z0-9\s.!?;:\'\"()\-\n]')
# WHITESPACE_RE = re.compile(r'\s+')

# # =========================
# # Cleaning
# # =========================

# def clean_article(text):
#     text = URL_RE.sub('', text)
#     text = SPECIAL_RE.sub('', text)
#     text = WHITESPACE_RE.sub(' ', text)
#     return text.strip()

# # =========================
# # Streaming Dataset
# # =========================

# dataset = load_dataset(
#     "wikimedia/wikipedia",
#     "20231101.en",
#     split="train",
#     streaming=True
# )

# # =========================
# # Output files
# # =========================

# train_path = os.path.join(dtst, "train.jsonl")
# val_path = os.path.join(dtst, "val.jsonl")
# test_path = os.path.join(dtst, "test.jsonl")

# train_file = open(train_path, "a", encoding="utf-8")
# val_file = open(val_path, "a", encoding="utf-8")
# test_file = open(test_path, "a", encoding="utf-8")

# try:
#     for i, row in enumerate(islice(dataset, 300_000), start=1):

#         cleaned = clean_article(row["text"])

#         if not cleaned:
#             continue

#         record = json.dumps(
#             {"text": cleaned},
#             ensure_ascii=False
#         ) + "\n"

#         r = i % 100

#         if r < 90:
#             train_file.write(record)
#         elif r < 95:
#             val_file.write(record)
#         else:
#             test_file.write(record)

#         if i % 10000 == 0:
#             print(f"Processed {i:,} articles")

# except KeyboardInterrupt:
#     print("Stopped by user")

# finally:
#     train_file.close()
#     val_file.close()
#     test_file.close()


In [ ]:
append_jsonl('train',flattened_dialogues['train'])
append_jsonl('test',flattened_dialogues['test'])
append_jsonl('val', flattened_dialogues['validation'])

Processed 50,000 articles
Processed 100,000 articles
Processed 150,000 articles
Processed 200,000 articles
Processed 250,000 articles
Processed 300,000 articles
Processed 350,000 articles
Processed 400,000 articles
Processed 450,000 articles


#Tokenization

In [ ]:
#Creating my Own Tokenizer based on my data (Run only once, don't run a lot)
drive_model = folder_path + '/my_tokenizer'
dtst = folder_path + '/Datasets'

spm.SentencePieceTrainer.train(
        input = dtst + '/train.jsonl',
        model_prefix = folder_path,
        vocab_size=8000,
        model_type='bpe',
        pad_id=0,
        unk_id=1,
        bos_id=-1,
        eos_id=-1
    )

In [ ]:
EOS_TOKEN = 99

def stream_jsonl_texts(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            text = (obj.get("text") or "").strip()
            if text:
                yield text

def tokenize_and_save(split_name):
    sp = spm.SentencePieceProcessor(model_file=folder_path + "/my_tokenizer.model")
    path = os.path.join(folder_path,"Datasets",f"{split_name}.jsonl")
    out_path = os.path.join(folder_path,"tokenized",f"{split_name}.bin")
    with open(out_path, "wb") as fout:
        for i, text in enumerate(stream_jsonl_texts(path), 1):
            ids = sp.encode(text, out_type=int)
            ids.append(EOS_TOKEN)
            arr = np.array(ids, dtype=np.uint16)
            arr.tofile(fout)
            if i % 50000 == 0:
                print(f"Tokenized {i:,} articles")
    print(f"Finished {split_name}")

def run_pipeline():

    tokenize_and_save("train")
    tokenize_and_save("val")
    tokenize_and_save("test")


# def tokenize_and_save(split_name):
#     i=0
#     sp = spm.SentencePieceProcessor(model_file = folder_path + '/my_tokenizer.model')

#     path = os.path.join(folder_path, "Datasets", f"{split_name}.jsonl")
#     texts = load_jsonl_texts(path)

#     token_ids = []

#     for text in texts:
#         ids = sp.encode(text, out_type=int)
#         token_ids.extend(ids)
#         token_ids.append(EOS_TOKEN)
#         i+=1
#         if i % 50000 == 0:
#           print(f"Tokenized {i:,} articles")

#     # convert to numpy array
#     token_array = np.array(token_ids, dtype=np.uint16)

#     # save as .bin
#     out_path = os.path.join(folder_path,"tokenized", f"{split_name}.bin")
#     token_array.tofile(out_path)

#     print(f"{split_name}: saved {len(token_array)} tokens → {out_path}")


In [ ]:
run_pipeline()

Finished val


##Creating Context Windows

In [ ]:
train_token = np.fromfile(folder_path + '/tokenized'+ "/val.bin", dtype=np.uint16)
len(train_token)

15634009

In [ ]:
# Defining the Context size for the Small Language Model. The context size is defined by how complex we want the model to be.

context_size = 512
total_length = (len(train_token) // context_size) * context_size

total_ids = []
for i in range(0, total_length, context_size):
    context = train_token[i : i + context_size]
    total_ids.append(context)

In [ ]:
len(total_ids)

30535

#TRANSFORMER

##Token Embedding

In [ ]:
# batch_size = 4

# def stream_and_save_numpy(data_chunks,batch_size,output_dir):

#     os.makedirs(output_dir, exist_ok=True)

#     # shuffled_chunks = data_chunks.copy()
#     # random.shuffle(shuffled_chunks)

#     total_samples = len(data_chunks)

#     x_path = os.path.join(output_dir, "X.dat")
#     y_path = os.path.join(output_dir, "Y.dat")

#     # Create disk-backed arrays
#     X_memmap = np.memmap(
#         x_path,
#         dtype=np.int32,
#         mode="w+",
#         shape=(total_samples, 511)
#     )

#     Y_memmap = np.memmap(
#         y_path,
#         dtype=np.int32,
#         mode="w+",
#         shape=(total_samples, 511)
#     )

#     write_pos = 0

#     print(f"Processing {total_samples:,} contexts")

#     for i in range(0, total_samples, batch_size):

#         batch_group = data_chunks[i:i + batch_size]

#         if len(batch_group) < batch_size:
#             break

#         X_batch = np.asarray(
#             [window[:-1] for window in batch_group],
#             dtype=np.int32
#         )

#         Y_batch = np.asarray(
#             [window[1:] for window in batch_group],
#             dtype=np.int32
#         )

#         end_pos = write_pos + len(batch_group)

#         X_memmap[write_pos:end_pos] = X_batch
#         Y_memmap[write_pos:end_pos] = Y_batch

#         write_pos = end_pos

#         if write_pos % 100000 == 0:
#             print(
#                 f"[PROGRESS] {write_pos:,} contexts processed"
#             )

#     X_memmap.flush()
#     Y_memmap.flush()

#     print(f"\nFinished.")
#     print(f"Saved {write_pos:,} samples")

#     return write_pos

In [ ]:
# output_dir = folder_path + "/token_embedded_val"

# total_saved = stream_and_save_numpy(total_ids,batch_size,output_dir)

Processing 30,535 contexts

Finished.
Saved 30,532 samples


## Rerun



In [ ]:

output_dir = folder_path + '/token_embedded'
num_samples = 554237

X = np.memmap(
    output_dir + "/X.dat",
    dtype=np.int32,
    mode="r",
    shape=(num_samples, 511)
)

x_tensor_train = torch.from_numpy(X)
x_tensor_train.shape

Y = np.memmap(
    output_dir + "/Y.dat",
    dtype=np.int32,
    mode="r",
    shape=(num_samples, 511)
)

y_tensor_train = torch.from_numpy(X)
y_tensor_train.shape

/tmp/ipykernel_256/188866783.py:11: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  x_tensor_train = torch.from_numpy(X)


torch.Size([554237, 511])

In [ ]:
# x_tensor_train = x_tensor_train[:1000]
# y_tensor_train = y_tensor_train[:1000]

In [ ]:
output_dir = folder_path + '/token_embedded_val'
num_samples = 30535

X = np.memmap(
    output_dir + "/X.dat",
    dtype=np.int32,
    mode="r",
    shape=(num_samples, 511)
)

x_tensor_val = torch.from_numpy(X)
x_tensor_val.shape

Y = np.memmap(
    output_dir + "/Y.dat",
    dtype=np.int32,
    mode="r",
    shape=(num_samples, 511)
)

y_tensor_val = torch.from_numpy(X)
y_tensor_val.shape

torch.Size([30535, 511])

In [ ]:
# x_tensor_val = x_tensor_val[:1000]
# y_tensor_val = y_tensor_val[:1000]

In [ ]:
vocab_size = 8000
d_model = 384

embedding_table = []
for _ in range(vocab_size):
    # Generate 384 random numbers for a single word profile
    word_vector = [random.uniform(-0.1, 0.1) for _ in range(d_model)]
    embedding_table.append(word_vector)

lookup_table = cp.array(embedding_table, dtype=np.float32)

In [ ]:
lookup_table.shape

(8000, 384)

##Layers of Transformer

In [ ]:
# class architecture:
#   def __init__(self, lookup_table, x_tensor, y_tensor, x_tensor_val, y_tensor_val):
#     self.vocab_size = 8000 #vocabulary size
#     self.batch_size = 64
#     self.d_model = 384 #number of vectors to define the word
#     self.total_samples = 554237

#     self.seq_len = 511 #Length of one tensor (x_tensor and y_tensor)
#     self.pos_emb_mat = cp.random.randn(self.seq_len, self.d_model).astype(cp.float32) * 0.01 #postional embedding absolute value

#     self.lookup_table = lookup_table.astype(cp.float32)

#     self.x_tensor = cp.array(np.from_dlpack(x_tensor))
#     self.y_tensor = cp.array(np.from_dlpack(y_tensor))
#     self.x_tensor_val = cp.array(np.from_dlpack(x_tensor_val))
#     self.y_tensor_val = cp.array(np.from_dlpack(y_tensor_val))

#     scale = 0.1
#     self.W_Q = cp.random.randn(self.d_model, self.d_model).astype(cp.float32) * scale
#     self.W_K = cp.random.randn(self.d_model, self.d_model).astype(cp.float32) * scale
#     self.W_V = cp.random.randn(self.d_model, self.d_model).astype(cp.float32) * scale
#     self.d_b_vocab = cp.zeros(self.vocab_size, dtype=cp.float32) #bias vector for lookup table
#     self.i = 1

#     # parameters for adamW
#     self.lr = 0.001
#     self.beta1 = 0.9
#     self.beta2 = 0.999
#     self.eps = 0.00000001
#     self.weight_decay = 0.01
#     self.t = 0
#     self.m_W_Q = cp.zeros_like(self.W_Q)
#     self.v_W_Q = cp.zeros_like(self.W_Q)
#     self.m_W_K = cp.zeros_like(self.W_K)
#     self.v_W_K = cp.zeros_like(self.W_K)
#     self.m_W_V = cp.zeros_like(self.W_V)
#     self.v_W_V = cp.zeros_like(self.W_V)
#     self.m_lookup = cp.zeros_like(self.lookup_table)
#     self.v_lookup = cp.zeros_like(self.lookup_table)
#     self.m_pos = cp.zeros_like(self.pos_emb_mat)
#     self.v_pos = cp.zeros_like(self.pos_emb_mat)
#     self.m_bias = cp.zeros_like(self.d_b_vocab)
#     self.v_bias = cp.zeros_like(self.d_b_vocab)
#     self.m_b_FF1 = cp.zeros_like(4*self.d_model)
#     self.v_b_FF1 = cp.zeros_like(4*self.d_model)
#     self.m_b_FF2 = cp.zeros_like(self.d_model)
#     self.v_b_FF2 = cp.zeros_like(self.d_model)

#     # casual masking
#     self.causal_mask = cp.tril(cp.ones((self.seq_len, self.seq_len), dtype=cp.float32))

#     # Gelu actiation fucntion weights
#     self.W_f1 = cp.random.randn(self.d_model, 4*self.d_model).astype(cp.float32) * scale
#     self.b_f1 = cp.random.randn(4*self.d_model).astype(cp.float32) * scale
#     self.W_f2 = cp.random.randn(4*self.d_model, self.d_model).astype(cp.float32) * scale
#     self.b_f2 = cp.random.randn(self.d_model).astype(cp.float32) * scale

#     self.gamma_ln = cp.ones(self.d_model, dtype=cp.float32)
#     self.beta_ln = cp.zeros(self.d_model, dtype=cp.float32)

#     self.m_gamma = cp.zeros_like(self.gamma_ln)
#     self.v_gamma = cp.zeros_like(self.gamma_ln)

#     self.m_beta = cp.zeros_like(self.beta_ln)
#     self.v_beta = cp.zeros_like(self.beta_ln)

#   def adamw_update(self, param, grad, m, v):
#     m = self.beta1 * m + (1 - self.beta1) * grad
#     v = self.beta2 * v + (1 - self.beta2) * (grad * grad)
#     m_hat = m / (1 - self.beta1 ** self.t)
#     v_hat = v / (1 - self.beta2 ** self.t)
#     param -= self.lr * m_hat / (cp.sqrt(v_hat) + self.eps)

#     # Decoupled weight decay (AdamW)
#     param -= self.lr * self.weight_decay * param

#     return param, m, v

#   def show(self):
#     print(self.lookup_table.shape)
#     print(self.x_tensor.shape)
#     print(self.y_tensor.shape)

#   def gelu(self):
#     hello_world

#   def stream(self, nig, ga):
#     x = len(nig)
#     for i in range(0, x, self.batch_size):
#       end_idx = min(i + self.batch_size, x)
#       batch_x = nig[i:end_idx]
#       batch_y = ga[i:end_idx]
#       yield batch_x, batch_y

#   def validation(self, batch_x, batch_y):
#     token_emb = self.lookup_table[batch_x] #Token Embedding

#     Z = token_emb + self.pos_emb_mat #Positional Embedding
#     mean = cp.mean(Z, axis=-1, keepdims=True)
#     var = cp.var(Z, axis=-1, keepdims=True)
#     Z_norm = (Z - mean) / cp.sqrt(var + 1e-5)

#     #Attention Mechanism
#     Q = cp.matmul(Z_norm, self.W_Q)
#     K = cp.matmul(Z_norm, self.W_K)
#     V = cp.matmul(Z_norm, self.W_V)

#     # scores = cp.matmul(Q, cp.swapaxes(K, 1, 2))
#     # scaled_scores = scores / cp.sqrt(self.

#     # scaled_scores = cp.matmul(Q, cp.swapaxes(K, 1, 2)) / cp.sqrt(self.d_model)
#     scaled_scores = cp.matmul(Q, cp.swapaxes(K, 1, 2)) / cp.sqrt(self.d_model)

#     # expand mask to batch dimension
#     mask = self.causal_mask[:batch_x.shape[1], :batch_x.shape[1]]
#     mask = mask[None, :, :]   # shape (1, T, T)

#     # convert mask to additive form
#     scaled_scores = cp.where(mask == 1, scaled_scores, -1e9)

#     exp_scores = cp.exp(scaled_scores - cp.max(scaled_scores, axis=-1, keepdims=True))
#     attention_weights = exp_scores / cp.sum(exp_scores, axis=-1, keepdims=True)

#     H = cp.matmul(attention_weights, V)
#     H = H + Z # Residual Connection

#     # Layer Norm (Pre-LN)
#     mean = cp.mean(H, axis=-1, keepdims=True)
#     var = cp.var(H, axis=-1, keepdims=True)

#     H_norm = (H - mean) / cp.sqrt(var + 1e-5)

#     H_norm = H_norm * self.gamma_ln + self.beta_ln


#     # Feed Forward Network (uses activation function of Gaussian Error Linear Unit)
#     f1 = cp.matmul(H_norm, self.W_f1) + self.b_f1
#     gelu_out = 0.5 * f1 * (1.0 + cp.tanh(cp.sqrt(2.0 / cp.pi) * (f1 + 0.044715 * cp.power(x, 3)))) # GELU
#     f2 = cp.matmul(gelu_out, self.W_f2) + self.b_f2
#     ffn_out = f2
#     H = H + ffn_out


#     # Calculating Loss
#     logits = cp.matmul(H, self.lookup_table.T)
#     exp_logits = cp.exp(logits - cp.max(logits, axis=-1, keepdims=True))
#     probabilities = exp_logits / cp.sum(exp_logits, axis=-1, keepdims=True)

#     targets = batch_y[..., None]
#     true_word_probs = cp.take_along_axis(probabilities, targets, axis=-1).squeeze(-1)

#     true_word_probs = cp.clip(true_word_probs, 1e-15, 1.0)
#     losses = -cp.log(true_word_probs)
#     mean_loss = cp.mean(losses)
#     # print(f"Train Batch {self.i} Loss: {mean_loss.item():.4f}")
#     # self.i+=1

#     return mean_loss

#   def train(self, batch_x, batch_y):
#     token_emb = self.lookup_table[batch_x] #Token Embedding

#     Z = token_emb + self.pos_emb_mat #Positional Embedding
#     mean = cp.mean(Z, axis=-1, keepdims=True)
#     var = cp.var(Z, axis=-1, keepdims=True)
#     Z_norm = (Z - mean) / cp.sqrt(var + 1e-5)

#     #Attention Mechanism
#     Q = cp.matmul(Z_norm, self.W_Q)
#     K = cp.matmul(Z_norm, self.W_K)
#     V = cp.matmul(Z_norm, self.W_V)

#     # scores = cp.matmul(Q, cp.swapaxes(K, 1, 2))
#     # scaled_scores = scores / cp.sqrt(self.d_model)

#     # scaled_scores = cp.matmul(Q, cp.swapaxes(K, 1, 2)) / cp.sqrt(self.d_model)
#     scaled_scores = cp.matmul(Q, cp.swapaxes(K, 1, 2)) / cp.sqrt(self.d_model)

#     # expand mask to batch dimension
#     mask = self.causal_mask[:batch_x.shape[1], :batch_x.shape[1]]
#     mask = mask[None, :, :]   # shape (1, T, T)

#     # convert mask to additive form
#     scaled_scores = cp.where(mask == 1, scaled_scores, -1e9)

#     exp_scores = cp.exp(scaled_scores - cp.max(scaled_scores, axis=-1, keepdims=True))
#     attention_weights = exp_scores / cp.sum(exp_scores, axis=-1, keepdims=True)

#     H = cp.matmul(attention_weights, V)
#     H = H + Z # Residual Connection

#     # Layer Norm (Pre-LN)
#     mean = cp.mean(H, axis=-1, keepdims=True)
#     var = cp.var(H, axis=-1, keepdims=True)

#     H_norm = (H - mean) / cp.sqrt(var + 1e-5)

#     H_norm = H_norm * self.gamma_ln + self.beta_ln


#     # Feed Forward Network (uses activation function of Gaussian Error Linear Unit)
#     f1 = cp.matmul(H_norm, self.W_f1) + self.b_f1
#     gelu_out = 0.5 * f1 * (1.0 + cp.tanh(cp.sqrt(2.0 / cp.pi) * (f1 + 0.044715 * cp.power(x, 3)))) # GELU
#     f2 = cp.matmul(gelu_out, self.W_f2) + self.b_f2
#     ffn_out = f2
#     H = H + ffn_out


#     # Calculating Loss
#     logits = cp.matmul(H, self.lookup_table.T)
#     exp_logits = cp.exp(logits - cp.max(logits, axis=-1, keepdims=True))
#     probabilities = exp_logits / cp.sum(exp_logits, axis=-1, keepdims=True)

#     targets = batch_y[..., None]
#     true_word_probs = cp.take_along_axis(probabilities, targets, axis=-1).squeeze(-1)

#     true_word_probs = cp.clip(true_word_probs, 1e-15, 1.0)
#     losses = -cp.log(true_word_probs)
#     mean_loss = cp.mean(losses)
#     print(f"Train Batch {self.i} Loss: {mean_loss.item():.4f}")
#     self.i+=1




#     # Backpropogation
#     # Step 1: partial differentiation of the loss wrt to logits
#     batch_idx = cp.arange(self.batch_size)[:, None]
#     time_idx = cp.arange(self.seq_len)[None, :]

#     probabilities[batch_idx, time_idx, batch_y] -= 1
#     probabilities /= (self.batch_size * self.seq_len)

#     d_logits = probabilities
#     d_f2 = d_logits

#     d_W_FF2 = cp.einsum('btd,btv->dv', gelu_out, d_f2)
#     d_b_FF2 = cp.sum(d_f2, axis=(0,1))
#     d_gelu_out = cp.matmul(d_f2, self.W_f2.T)

#     gelu_grad = 0.5 * (1 + cp.tanh(cp.sqrt(2.0 / cp.pi) * (f1 + 0.044715 * f1**3))) \
#     + 0.5 * f1 * (1 - cp.tanh(cp.sqrt(2.0 / cp.pi) * (f1 + 0.044715 * f1**3))**2) \
#     * cp.sqrt(2.0 / cp.pi) * (1 + 3 * 0.044715 * f1**2)

#     d_f1 = d_gelu_out * gelu_grad
#     d_W_FF1 = cp.einsum('btd,btv->dv', H, d_f1)
#     d_b_FF1 = cp.sum(d_f1, axis=(0,1))
#     d_H_ffn = cp.matmul(d_f1, self.W_f1.T)

#     # Through output projection
#     d_w_vocab = cp.einsum('btv,btd->vd', d_logits, H)

#     d_b_vocab = cp.sum(d_logits, axis=(0,1))
#     d_hidden = cp.matmul(d_logits, self.lookup_table)
#     d_H = d_hidden + d_H_ffn

#     d_gamma = cp.sum(d_H * H_norm, axis=(0,1))
#     d_beta = cp.sum(d_H, axis=(0,1))

#     d_H_norm = d_H * self.gamma_ln


#     #gradient wrt attention weights
#     d_V = cp.matmul(cp.swapaxes(attention_weights, 1, 2), d_hidden)

#     d_w_V = cp.einsum('btd,btv->dv', Z_norm, d_V) Z

#     d_attn = cp.matmul(d_hidden, cp.swapaxes(V, 1, 2))

#     d_scaled_scores = attention_weights * (d_attn - cp.sum(d_attn * attention_weights, axis=-1, keepdims=True))

#     d_Q = cp.matmul(d_scaled_scores,K)
#     d_K = cp.matmul(cp.swapaxes(d_scaled_scores, 1, 2), Q)


#     d_W_Q = cp.einsum('btd,btv->dv', Z_norm, d_Q)
#     d_W_K = cp.einsum('btd,btv->dv', Z_norm, d_K)
#     d_W_V = cp.einsum('btd,btv->dv', Z_norm, d_V)

#     clip = 1.0

#     d_W_Q *= min(1, clip/(cp.linalg.norm(d_W_Q)+1e-8))
#     d_W_K *= min(1, clip/(cp.linalg.norm(d_W_K)+1e-8))
#     d_W_V *= min(1, clip/(cp.linalg.norm(d_W_V)+1e-8))

#     d_Z = (cp.matmul(d_Q, self.W_Q.T) + cp.matmul(d_K, self.W_K.T) + cp.matmul(d_V, self.W_V.T))

#     #for the first use of lookup_table
#     d_lookup_table_input = cp.zeros_like(self.lookup_table)

#     cp.add.at(d_lookup_table_input,batch_x,d_Z / self.seq_len)

#     #Updating the parameters
#     learning_rate = 0.9
#     beta_1 = 0.9 # momentum
#     beta_2 = 0.999
#     ep = 0.00000001
#     weight_d = 0.01 # weight decay

#     # self.W_Q -= learning_rate * d_W_Q
#     # self.W_K -= learning_rate * d_W_K
#     # self.W_V -= learning_rate * d_W_V

#     total_lookup_gradient = d_w_vocab + d_lookup_table_input

#     # self.lookup_table -= learning_rate * total_lookup_gradient
#     # self.d_b_vocab -= learning_rate * d_b_vocab

#     d_pos_emb = cp.sum(d_Z, axis=0)
#     # self.pos_emb_mat -= learning_rate * d_pos_emb
#     self.t += 1
#     self.W_Q, self.m_W_Q, self.v_W_Q = self.adamw_update(self.W_Q,d_W_Q,self.m_W_Q,self.v_W_Q)
#     self.W_K, self.m_W_K, self.v_W_K = self.adamw_update(self.W_K,d_W_K,self.m_W_K,self.v_W_K)
#     self.W_V, self.m_W_V, self.v_W_V = self.adamw_update(self.W_V,d_W_V,self.m_W_V,self.v_W_V)
#     self.lookup_table, self.m_lookup, self.v_lookup = self.adamw_update(self.lookup_table,total_lookup_gradient,self.m_lookup,self.v_lookup)
#     self.pos_emb_mat, self.m_pos, self.v_pos = self.adamw_update(self.pos_emb_mat,d_pos_emb,self.m_pos,self.v_pos)
#     self.d_b_vocab, self.m_bias, self.v_bias = self.adamw_update(self.d_b_vocab,d_b_vocab,self.m_bias,self.v_bias)

#     self.b_f1, self.m_b_FF1, self.v_b_FF1 = self.adamw_update(self.b_f1, d_b_FF1, self.m_b_FF1, self.v_b_FF1)
#     self.b_f2, self.m_b_FF2, self.v_b_FF2 = self.adamw_update(self.b_f2, d_b_FF2, self.m_b_FF2, self.v_b_FF2)

#     self.gamma_ln, self.m_gamma, self.v_gamma = self.adamw_update(self.gamma_ln, d_gamma, self.m_gamma, self.v_gamma)
#     self.beta_ln, self.m_beta, self.v_beta = self.adamw_update(self.beta_ln, d_beta, self.m_beta, self.v_beta)

#     return mean_loss


#     # print("t =", self.t)

#     # print("d_W_Q norm =", cp.linalg.norm(d_W_Q))
#     # print("d_W_K norm =", cp.linalg.norm(d_W_K))
#     # print("d_W_V norm =", cp.linalg.norm(d_W_V))

#     # print("m_W_Q norm =", cp.linalg.norm(self.m_W_Q))
#     # print("v_W_Q norm =", cp.linalg.norm(self.v_W_Q))

#     # print("d_w_vocab = ",cp.linalg.norm(d_w_vocab))
#     # print("lookup_table = ",cp.linalg.norm(d_lookup_table_input))

#     # print("\n\n")

#     # # To check for vanishing gradient
#     # print("d_logits: ", cp.linalg.norm(d_logits))
#     # print("d_W_Q: ", cp.linalg.norm(d_W_Q))
#     # print("d_W_K: ", cp.linalg.norm(d_W_K))
#     # print("d_W_V: ", cp.linalg.norm(d_W_V))
#     # print("d_scaled_scores: ", cp.linalg.norm(d_scaled_scores))
#     # print("total_lookup_gradient: ", cp.linalg.norm(total_lookup_gradient))
#     # print("d_pos_emb: ", cp.linalg.norm(d_pos_emb))
#     # print("\n")

#     # print(cp.std(scaled_scores))
#     # print(cp.std(attention_weights))

#     # print("Q std:", cp.std(Q))
#     # print("K std:", cp.std(K))
#     # print("V std:", cp.std(V))

#     # print("scaled_scores std:", cp.std(scaled_scores))
#     # print("attention_weights std:", cp.std(attention_weights))
#     # print("\n")
#     # print("\n")



#     # print("SHAPES OF DIFFERENT THINGS")
#     # print("Z: ", Z.shape)
#     # print("Lookup: ", self.lookup_table.shape)
#     # print("probabilities: ",probabilities.shape)
#     # print("true word prob: ",true_word_probs.shape)
#     # print("losses: ",losses.shape)
#     # print("d_logits: ", d_logits.shape)
#     # print("d_w_vocab: ", d_w_vocab.shape)
#     # print("d_b_vocab: ", d_b_vocab.shape)
#     # print("d_hidden: ", d_hidden.shape)
#     # print("d_V: ", d_V.shape)
#     # print("d_w_V: ", d_w_V.shape)
#     # print("d_attn_w: ", d_attn_w.shape)
#     # print("d_scaled_scores: ", d_scaled_scores.shape)
#     # print("d_Q: ", d_Q.shape)
#     # print("d_K: ", d_K.shape)
#     # print("d_Z: ", d_Z.shape)
#     # print("d_W_Q: ", d_W_Q.shape)
#     # print("d_W_K: ", d_W_K.shape)
#     # print("d_W_V: ", d_W_V.shape)
#     # print("d_b_vocab: ", self.d_b_vocab.shape)

#     # break

#   # def epoch(self, it = 3):
#   #   for epoch in range(it):
#   #     print(f"Epoch {epoch+1}")

#   #     train_stream = self.stream(self.x_tensor, self.y_tensor)
#   #     val_stream = self.stream(self.x_tensor_val, self.y_tensor_val)

#   #     for (batch_x, batch_y), (batch_x_val, batch_y_val) in zip(train_stream, val_stream):
#   #         if batch_x.shape[0] == self.batch_size:
#   #             self.train(batch_x, batch_y)
#   #             self.validation(batch_x_val, batch_y_val)
#   #     print("Epoch complete\n")

#   def epoch(self, it=3):
#     for epoch in range(it):
#       print(f"Epoch {epoch + 1}")

#       # Training
#       # train_stream = self.stream(self.x_tensor, self.y_tensor)

#       train_loss = 0.0
#       train_batches = 0

#       for batch_x, batch_y in self.stream(self.x_tensor, self.y_tensor):
#         if (len(batch_x) != self.batch_size):
#           continue
#         loss = self.train(batch_x, batch_y)   # assume returns batch loss

#         train_loss += loss
#         train_batches += 1

#       avg_train_loss = train_loss / train_batches

#       # Validation
#       # val_stream = self.stream(self.x_tensor_val, self.y_tensor_val)

#       val_loss = 0.0
#       val_batches = 0

#       for batch_x_val, batch_y_val in self.stream(self.x_tensor_val, self.y_tensor_val):
#         if (len(batch_x_val) != self.batch_size):
#           continue
#         loss = self.validation(batch_x_val, batch_y_val)  # assume returns batch loss

#         val_loss += loss
#         val_batches += 1

#       avg_val_loss = val_loss / val_batches

#       print(
#           f"Train Loss: {avg_train_loss:.4f} \t\t"
#           f"Val Loss: {avg_val_loss:.4f}"
#       )
#       print("Epoch complete\n")



##NEW

In [ ]:
class arch:
  def __init__(self, lookup_table, x_tensor, y_tensor, x_tensor_val, y_tensor_val, attn_heads=4):
    self.vocab_size = 8000 #vocabulary size
    self.batch_size = 64
    self.d_model = 384 #number of vectors to define the word
    self.total_samples = 554237
    self.num_heads = attn_heads
    self.head_dim = self.d_model // self.num_heads

    self.seq_len = 511 #Length of one tensor (x_tensor and y_tensor)
    self.pos_emb_mat = cp.random.randn(self.seq_len, self.d_model).astype(cp.float32) * 0.01 #postional embedding absolute value

    self.lookup_table = lookup_table.astype(cp.float32)
    self.d_b_vocab = cp.zeros(self.vocab_size, dtype=cp.float32)

    self.x_tensor = cp.array(np.from_dlpack(x_tensor))
    self.y_tensor = cp.array(np.from_dlpack(y_tensor))
    self.x_tensor_val = cp.array(np.from_dlpack(x_tensor_val))
    self.y_tensor_val = cp.array(np.from_dlpack(y_tensor_val))

    scale = 0.1
    self.W_Q = []
    self.W_K = []
    self.W_V = []

    for i in range(self.num_heads):

      self.W_Q.append(
          cp.random.randn(
              self.d_model,
              self.head_dim
          ).astype(cp.float32) * scale
      )

      self.W_K.append(
          cp.random.randn(
              self.d_model,
              self.head_dim
          ).astype(cp.float32) * scale
      )

      self.W_V.append(
          cp.random.randn(
              self.d_model,
              self.head_dim
          ).astype(cp.float32) * scale
      )
    self.WO = cp.random.randn(self.num_heads * self.head_dim, self.d_model) * 0.02

    self.i = 1

    # parameters for adamW
    self.lr = 0.001
    self.beta1 = 0.9
    self.beta2 = 0.999
    self.eps = 0.00000001
    self.weight_decay = 0.01
    self.t = 0
    self.m_W_Q = cp.zeros_like(self.W_Q)
    self.v_W_Q = cp.zeros_like(self.W_Q)
    self.m_W_K = cp.zeros_like(self.W_K)
    self.v_W_K = cp.zeros_like(self.W_K)
    self.m_W_V = cp.zeros_like(self.W_V)
    self.v_W_V = cp.zeros_like(self.W_V)
    self.m_lookup = cp.zeros_like(self.lookup_table)
    self.v_lookup = cp.zeros_like(self.lookup_table)
    self.m_pos = cp.zeros_like(self.pos_emb_mat)
    self.v_pos = cp.zeros_like(self.pos_emb_mat)
    self.m_bias = cp.zeros_like(self.d_b_vocab)
    self.v_bias = cp.zeros_like(self.d_b_vocab)
    self.m_b_FF1 = cp.zeros_like(4*self.d_model)
    self.v_b_FF1 = cp.zeros_like(4*self.d_model)
    self.m_b_FF2 = cp.zeros_like(self.d_model)
    self.v_b_FF2 = cp.zeros_like(self.d_model)

    # casual masking
    self.causal_mask = cp.tril(cp.ones((self.seq_len, self.seq_len), dtype=cp.float32))

    # Gelu actiation fucntion weights
    self.W_f1 = cp.random.randn(self.d_model, 4*self.d_model).astype(cp.float32) * scale
    self.b_f1 = cp.random.randn(4*self.d_model).astype(cp.float32) * scale
    self.W_f2 = cp.random.randn(4*self.d_model, self.d_model).astype(cp.float32) * scale
    self.b_f2 = cp.random.randn(self.d_model).astype(cp.float32) * scale

    self.gamma_ln = cp.ones(self.d_model, dtype=cp.float32)
    self.beta_ln = cp.zeros(self.d_model, dtype=cp.float32)

    self.m_gamma = cp.zeros_like(self.gamma_ln)
    self.v_gamma = cp.zeros_like(self.gamma_ln)

    self.m_beta = cp.zeros_like(self.beta_ln)
    self.v_beta = cp.zeros_like(self.beta_ln)

  def adamw_update(self, param, grad, m, v):
    m = self.beta1 * m + (1 - self.beta1) * grad
    v = self.beta2 * v + (1 - self.beta2) * (grad * grad)
    m_hat = m / (1 - self.beta1 ** self.t)
    v_hat = v / (1 - self.beta2 ** self.t)
    param -= self.lr * m_hat / (cp.sqrt(v_hat) + self.eps)
    param -= self.lr * self.weight_decay * param

    return param, m, v

  def show(self):
    print(self.lookup_table.shape)
    print(self.x_tensor.shape)
    print(self.y_tensor.shape)

  def stream(self, nig, ga):
    x = len(nig)
    for i in range(0, x, self.batch_size):
      end_idx = min(i + self.batch_size, x)
      batch_x = nig[i:end_idx]
      batch_y = ga[i:end_idx]
      yield batch_x, batch_y


  def token_position_embed(self, batch_x, batch_y):
    token_emb = self.lookup_table[batch_x] #Token Embedding

    Z = token_emb + self.pos_emb_mat #Positional Embedding
    mean = cp.mean(Z, axis=-1, keepdims=True)
    var = cp.var(Z, axis=-1, keepdims=True)
    Z_norm = (Z - mean) / cp.sqrt(var + 1e-5)

    return Z_norm, Z

  def attention(self, Z_norm, i):
    Q = cp.matmul(Z_norm, self.W_Q[i])
    K = cp.matmul(Z_norm, self.W_K[i])
    V = cp.matmul(Z_norm, self.W_V[i])
    scaled_scores = cp.matmul(Q, cp.swapaxes(K, 1, 2)) / cp.sqrt(self.d_model)

    return scaled_scores, Q, K, V

  def casual_masking(self, scaled_scores, batch_x):
    mask = self.causal_mask[:batch_x.shape[1], :batch_x.shape[1]]
    mask = mask[None, :, :]
    scaled_scores = cp.where(mask == 1, scaled_scores, -1e9)
    return scaled_scores

  def softmax(self, scaled_scores):
    exp_scores = cp.exp(scaled_scores - cp.max(scaled_scores, axis=-1, keepdims=True))
    return exp_scores / cp.sum(exp_scores, axis=-1, keepdims=True)

  def multihead_attention(self, Z_norm, batch_x):
    head = []
    all_Q = []
    all_K = []
    all_V = []
    all_scores = []

    for i in range(self.num_heads):
      scaled_scores, Q, K, V = self.attention(Z_norm, i)
      mask = self.casual_masking(scaled_scores, batch_x)
      weights = self.softmax(mask)
      H_i = cp.matmul(weights, V)

      head.append(H_i)
      all_Q.append(Q)
      all_K.append(K)
      all_V.append(V)
      all_scores.append(scaled_scores)
    H_concat = cp.concatenate(head, axis=2)
    H = cp.matmul(H_concat, self.WO)

    return H, all_scores, all_Q, all_K, all_V

  def residual_connection(self, attention_weights, Z, V):
    H = H + Z
    return H

  def layer_norm(self, H):
    mean = cp.mean(H, axis=-1, keepdims=True)
    var = cp.var(H, axis=-1, keepdims=True)
    H_norm = (H - mean) / cp.sqrt(var + 1e-5)
    H_norm = H_norm * self.gamma_ln + self.beta_ln

    return H_norm


  def gelu(self, x):
     return 0.5 * x * (1.0 + cp.tanh(cp.sqrt(2.0 / cp.pi) * (x + 0.044715 * cp.power(x, 3))))

  def gelu_grad_cal(self, x):
    return 0.5 * (1 + cp.tanh(cp.sqrt(2.0 / cp.pi) * (x + 0.044715 * x**3))) \
    + 0.5 * f1 * (1 - cp.tanh(cp.sqrt(2.0 / cp.pi) * (x + 0.044715 * x**3))**2) \
    * cp.sqrt(2.0 / cp.pi) * (1 + 3 * 0.044715 * x**2)

  def feedforward_network(self, H_norm, H):
    f1 = cp.matmul(H_norm, self.W_f1) + self.b_f1
    gelu_out = self.gelu(f1)
    f2 = cp.matmul(gelu_out, self.W_f2) + self.b_f2
    ffn_out = f2
    H = H + ffn_out

    return f1, gelu_out, f2, H

  def calculating_loss(self, H, batch_y):
    logits = cp.matmul(H, self.lookup_table.T)
    probabilities = self.softmax(logits)

    targets = batch_y[..., None]
    true_word_probs = cp.take_along_axis(probabilities, targets, axis=-1).squeeze(-1)

    true_word_probs = cp.clip(true_word_probs, 1e-15, 1.0)
    losses = -cp.log(true_word_probs)
    mean_loss = cp.mean(losses)
    print(f"Train Batch {self.i} Loss: {mean_loss.item():.4f}")
    self.i+=1

    return probabilities

  def gradient_logits(self, probabilities, batch_y):
    one_hot_target = cp.zeros_like(probabilities)
    cp.put_along_axis(one_hot_target, batch_y[..., None], 1.0, axis=-1)
    N = batch_y.size
    d_logits = (probabilities - one_hot_target) / N
    return d_logits

  def gradient_wrt_ffn(self, d_logits, gelu_out, f1, H):
    # d_logits = probabilities
    d_f2 = d_logits

    d_W_FF2 = cp.einsum('btd,btv->dv', gelu_out, d_f2)
    d_b_FF2 = cp.sum(d_f2, axis=(0,1))
    d_gelu_out = cp.matmul(d_f2, self.W_f2.T)

    gelu_grad = self.gelu_grad_cal(f1)

    d_f1 = d_gelu_out * gelu_grad
    d_W_FF1 = cp.einsum('btd,btv->dv', H, d_f1)
    d_b_FF1 = cp.sum(d_f1, axis=(0,1))
    d_H_ffn = cp.matmul(d_f1, self.W_f1.T)

    return d_H_ffn, d_W_FF1, d_b_FF1, d_W_FF2, d_b_FF2

  def gradient_wrt_yomom(self, d_logits ,H , d_H_ffn):
    d_w_vocab = cp.einsum('btv,btd->vd', d_logits, H)
    d_b_vocab = cp.sum(d_logits, axis=(0,1))
    d_hidden = cp.matmul(d_logits, self.lookup_table)
    d_H = d_hidden + d_H_ffn
    d_gamma = cp.sum(d_H * H_norm, axis=(0,1))
    d_beta = cp.sum(d_H, axis=(0,1))
    d_H_norm = d_H * self.gamma_ln

    return d_H_norm, d_hidden, d_gamma, d_beta, d_w_vocab, d_b_vocab

  def gradient_wrt_attn(self ,attention_weights ,d_hidden ,Z_norm ,Q ,K ,V):
    d_V = cp.matmul(cp.swapaxes(attention_weights, 1, 2), d_hidden)
    d_w_V = cp.einsum('btd,btv->dv', Z_norm, d_V)
    d_attn = cp.matmul(d_hidden, cp.swapaxes(V, 1, 2))
    d_scaled_scores = attention_weights * (d_attn - cp.sum(d_attn * attention_weights, axis=-1, keepdims=True))
    d_Q = cp.matmul(d_scaled_scores,K)
    d_K = cp.matmul(cp.swapaxes(d_scaled_scores, 1, 2), Q)

    d_W_Q = cp.einsum('btd,btv->dv', Z_norm, d_Q)
    d_W_K = cp.einsum('btd,btv->dv', Z_norm, d_K)
    d_W_V = cp.einsum('btd,btv->dv', Z_norm, d_V)

    clip = 1.0

    d_W_Q *= min(1, clip/(cp.linalg.norm(d_W_Q)+1e-8))
    d_W_K *= min(1, clip/(cp.linalg.norm(d_W_K)+1e-8))
    d_W_V *= min(1, clip/(cp.linalg.norm(d_W_V)+1e-8))

    d_Z = (cp.matmul(d_Q, self.W_Q.T) + cp.matmul(d_K, self.W_K.T) + cp.matmul(d_V, self.W_V.T))

    return d_W_Q, d_W_K, d_W_V, d_Z

  def gradient_wrt_lookup(self, batch_x, d_Z, d_w_vocab):
    d_lookup_table_input = cp.zeros_like(self.lookup_table)
    cp.add.at(d_lookup_table_input,batch_x,d_Z / self.seq_len)
    total_lookup_gradient = d_w_vocab + d_lookup_table_input

    return total_lookup_gradient

  def weight_updation(self, d_W_Q, d_W_K, d_W_V, total_lookup_gradient, d_b_vocab, d_pos_emb, d_b_FF1, d_b_FF2, d_gamma, d_beta):
    learning_rate = 0.9
    beta_1 = 0.9
    beta_2 = 0.999
    ep = 0.00000001
    weight_d = 0.01

    self.t += 1
    self.W_Q, self.m_W_Q, self.v_W_Q = self.adamw_update(self.W_Q,d_W_Q,self.m_W_Q,self.v_W_Q)
    self.W_K, self.m_W_K, self.v_W_K = self.adamw_update(self.W_K,d_W_K,self.m_W_K,self.v_W_K)
    self.W_V, self.m_W_V, self.v_W_V = self.adamw_update(self.W_V,d_W_V,self.m_W_V,self.v_W_V)
    self.lookup_table, self.m_lookup, self.v_lookup = self.adamw_update(self.lookup_table,total_lookup_gradient,self.m_lookup,self.v_lookup)
    self.pos_emb_mat, self.m_pos, self.v_pos = self.adamw_update(self.pos_emb_mat,d_pos_emb,self.m_pos,self.v_pos)
    self.d_b_vocab, self.m_bias, self.v_bias = self.adamw_update(self.d_b_vocab,d_b_vocab,self.m_bias,self.v_bias)

    self.b_f1, self.m_b_FF1, self.v_b_FF1 = self.adamw_update(self.b_f1, d_b_FF1, self.m_b_FF1, self.v_b_FF1)
    self.b_f2, self.m_b_FF2, self.v_b_FF2 = self.adamw_update(self.b_f2, d_b_FF2, self.m_b_FF2, self.v_b_FF2)

    self.gamma_ln, self.m_gamma, self.v_gamma = self.adamw_update(self.gamma_ln, d_gamma, self.m_gamma, self.v_gamma)
    self.beta_ln, self.m_beta, self.v_beta = self.adamw_update(self.beta_ln, d_beta, self.m_beta, self.v_beta)


  def backpropogation(self, batch_x, probabilites, gelu_out, f1, H, attention_weights, Z_norm, Q, K, V):
    batch_idx = cp.arange(self.batch_size)[:, None]
    time_idx = cp.arange(self.seq_len)[None, :]

    probabilities[batch_idx, time_idx, batch_y] -= 1
    probabilities /= (self.batch_size * self.seq_len)

    d_H_ffn, d_W_FF1, d_b_FF1, d_W_FF2, d_b_FF2 = self.gradient_wrt_ffn(probabilities, gelu_out, f1, H)
    d_H_norm, d_hidden, d_gamma, d_beta, d_w_vocab, d_b_vocab = self.gradient_wrt_yomom(probabilites, H, d_H_ffn)
    d_W_Q, d_W_K, d_W_V, d_Z = self.gradient_wrt_attn(attention_weights, d_hidden, Z_norm, Q, K, V)
    total_lookup_gradient = self.gradient_wrt_lookup(batch_x, d_Z, d_w_vocab)
    d_pos_emb = cp.sum(d_Z, axis=0)
    self.weight_updation(d_W_Q, d_W_K, d_W_V, total_lookup_gradient, d_b_vocab, d_pos_emb, d_b_FF1, d_b_FF2, d_gamma, d_beta)


  def train(self, batch_x, batch_y):
    Z_norm, Z = self.token_position_embed(batch_x, batch_y)
    H, all_scores, all_Q, all_K, all_V = self.multihead_attention(self, Z_norm, batch_x)
    H = self.residual_connection(self, H, self.attention_weights, Z, V)
    H_norm = self.layer_norm(H)
    f1, gelu_out, f2, H = self.feedforward_network(H_norm, H)
    probabilites = self.calculating_loss(H, batch_y)
    self.backpropogation(batch_x, probabilites, gelu_out, f1, H, self.attention_weights, Z_norm, Q, K, V)

  def validation(self, batch_x, batch_y):
    Z_norm, Z = self.token_position_embed(batch_x, batch_y)
    scaled_scores, Q, K, V = self.attention(Z_norm)
    scaled_scores = self.casual_masking(scaled_scores, batch_x)
    attention_weights = self.softmax(scaled_scores)
    H = self.residual_connection(self, H, attention_weights, Z, V)
    H_norm = self.layer_norm(H)
    f1, gelu_out, f2, H = self.feedforward_network(H_norm, H)
    probabilites = self.calculating_loss(H, batch_y)

  def epoch(self, num_epochs):
    for epoch in range(num_epochs):
        print(f"\n========== Epoch {epoch+1} ==========\n")
        train_loss_sum = 0.0
        train_batches = 0
        for batch_x, batch_y in self.stream(self.x_tensor,self.y_tensor):
            if batch_x.shape[0] != self.batch_size:
                continue

            loss = self.train(batch_x, batch_y)
            train_loss_sum += float(loss)
            train_batches += 1

        avg_train_loss = (train_loss_sum / train_batches if train_batches > 0 else 0)


        val_loss_sum = 0.0
        val_batches = 0

        for batch_x_val, batch_y_val in self.stream(self.x_tensor_val,self.y_tensor_val):
            if batch_x_val.shape[0] != self.batch_size:
                continue

            loss = self.validation(batch_x_val,batch_y_val)
            val_loss_sum += float(loss)
            val_batches += 1

        avg_val_loss = (val_loss_sum / val_batches if val_batches > 0 else 0)

        print(f"Train Loss = {avg_train_loss:.4f}, \t\t"f"Val Loss = {avg_val_loss:.4f}")

  def sell():
    for batch_x, batch_y in self.stream(self.x_tensor,self.y_tensor):
      print(batch_x.shape)
      return 23

In [ ]:
nig = arch(lookup_table, x_tensor_train, y_tensor_train, x_tensor_val, y_tensor_val)
# nig.show()
nig.sell()

AttributeError: 'list' object has no attribute 'dtype'